# Boop — AlphaZero Training

Trains a single **policy + value network** guided by **MCTS self-play**,
following the AlphaZero approach.

### Why AlphaZero instead of DQN?

| | DQN | AlphaZero |
|---|---|---|
| Lookahead | 1-step Q-value | MCTS tree search |
| Networks | 2 separate (one per player) | 1 shared (perspective-agnostic) |
| Exploration | epsilon-greedy (random) | UCT + Dirichlet noise (principled) |
| Targets | TD bootstrapping | MCTS visit counts + game outcome |

### Training loop
1. **Self-play**: each move uses MCTS (`MCTS_SIMS` simulations) guided by the current network
2. **Targets collected**: (observation, MCTS policy, game outcome) per move
3. **Train**: minimize cross-entropy(policy head, MCTS policy) + MSE(value head, outcome)
4. **Repeat** — the network improves, making future MCTS searches stronger

### Metrics printed every `EVAL_EVERY` episodes
- `p_loss` — policy head cross-entropy vs MCTS target
- `v_loss` — value head MSE vs actual game outcome
- `vs_random` — win rate of the greedy policy vs random (no MCTS at eval time)
- `vs_prev` — win rate vs previous checkpoint
- `ELO` — estimate anchored at 600 for a random agent

In [ ]:
%pip install open_spiel -q

In [ ]:
# Boop board game -- inlined for Colab (matches open_spiel/python/games/boop.py)
# Action encoding: piece_type * 36 + row * 6 + col  (72 actions total)
# Observation tensor: 5 board planes (6x6) + 4 hand scalars = 184 floats

import numpy as np
from open_spiel.python.observation import IIGObserverForPublicInfoGame
import pyspiel

_NUM_PLAYERS = 2
_ROWS = 6
_COLS = 6
_NUM_CELLS = _ROWS * _COLS
_NUM_PIECE_TYPES = 2
_NUM_ACTIONS = _NUM_PIECE_TYPES * _NUM_CELLS  # 72
_MAX_KITTENS = 8
_MAX_CATS = 6
_MAX_GAME_LENGTH = 500

_EMPTY = 0
_P0_KITTEN = 1
_P0_CAT = 2
_P1_KITTEN = 3
_P1_CAT = 4

_KITTEN_VAL = [_P0_KITTEN, _P1_KITTEN]
_CAT_VAL = [_P0_CAT, _P1_CAT]
_PIECE_VALS = [[_P0_KITTEN, _P0_CAT], [_P1_KITTEN, _P1_CAT]]

_GAME_TYPE = pyspiel.GameType(
    short_name='python_boop',
    long_name='Python Boop',
    dynamics=pyspiel.GameType.Dynamics.SEQUENTIAL,
    chance_mode=pyspiel.GameType.ChanceMode.DETERMINISTIC,
    information=pyspiel.GameType.Information.PERFECT_INFORMATION,
    utility=pyspiel.GameType.Utility.ZERO_SUM,
    reward_model=pyspiel.GameType.RewardModel.TERMINAL,
    max_num_players=_NUM_PLAYERS,
    min_num_players=_NUM_PLAYERS,
    provides_information_state_string=True,
    provides_information_state_tensor=False,
    provides_observation_string=True,
    provides_observation_tensor=True,
    parameter_specification={})

_GAME_INFO = pyspiel.GameInfo(
    num_distinct_actions=_NUM_ACTIONS,
    max_chance_outcomes=0,
    num_players=_NUM_PLAYERS,
    min_utility=-1.0,
    max_utility=1.0,
    utility_sum=0.0,
    max_game_length=_MAX_GAME_LENGTH)


class BoopGame(pyspiel.Game):
  def __init__(self, params=None):
    super().__init__(_GAME_TYPE, _GAME_INFO, params or dict())

  def new_initial_state(self):
    return BoopState(self)

  def make_py_observer(self, iig_obs_type=None, params=None):
    if ((iig_obs_type is None) or
        (iig_obs_type.public_info and not iig_obs_type.perfect_recall)):
      return BoopObserver(params)
    return IIGObserverForPublicInfoGame(iig_obs_type, params)


class BoopState(pyspiel.State):
  def __init__(self, game):
    super().__init__(game)
    self._cur_player = 0
    self._is_terminal = False
    self._winner = None
    self._move_count = 0
    self.board = np.zeros((_ROWS, _COLS), dtype=np.int8)
    self._hand = [[_MAX_KITTENS, 0], [_MAX_KITTENS, 0]]

  def current_player(self):
    return pyspiel.PlayerId.TERMINAL if self._is_terminal else self._cur_player

  def _legal_actions(self, player):
    actions = []
    for r in range(_ROWS):
      for c in range(_COLS):
        if self.board[r, c] == _EMPTY:
          cell = r * _COLS + c
          if self._hand[player][0] > 0:
            actions.append(cell)
          if self._hand[player][1] > 0:
            actions.append(_NUM_CELLS + cell)
    return sorted(actions)

  def _apply_action(self, action):
    piece_type = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    p = self._cur_player
    self._hand[p][piece_type] -= 1
    self.board[r, c] = _PIECE_VALS[p][piece_type]
    self._boop(r, c, is_cat=(piece_type == 1))
    self._move_count += 1
    if self._move_count >= _MAX_GAME_LENGTH:
      self._is_terminal = True
      return
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._promote_kittens(p)
    self._promote_kittens(1 - p)
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._cur_player = 1 - p
    # If next player has no moves (all pieces on board, none in hand), declare draw
    if not self._legal_actions(self._cur_player):
      self._is_terminal = True

  def _action_to_string(self, player, action):
    pt = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    piece = 'cat' if pt else 'kitten'
    return f'p{player}:{piece}@({r},{c})'

  def is_terminal(self):
    return self._is_terminal

  def returns(self):
    if self._winner == 0:
      return [1.0, -1.0]
    if self._winner == 1:
      return [-1.0, 1.0]
    return [0.0, 0.0]

  def __str__(self):
    syms = {
        _EMPTY: '.', _P0_KITTEN: 'k', _P0_CAT: 'K',
        _P1_KITTEN: 'o', _P1_CAT: 'O',
    }
    rows = [
        ''.join(syms[self.board[r, c]] for c in range(_COLS))
        for r in range(_ROWS)
    ]
    rows.append(
        f'P0: {self._hand[0][0]}k {self._hand[0][1]}K  '
        f'P1: {self._hand[1][0]}k {self._hand[1][1]}K  '
        f'move={self._move_count}')
    return '\n'.join(rows)

  def _boop(self, r, c, is_cat):
    for dr in (-1, 0, 1):
      for dc in (-1, 0, 1):
        if dr == 0 and dc == 0:
          continue
        nr, nc = r + dr, c + dc
        if not (0 <= nr < _ROWS and 0 <= nc < _COLS):
          continue
        neighbor = self.board[nr, nc]
        if neighbor == _EMPTY:
          continue
        neighbor_is_cat = neighbor in (_P0_CAT, _P1_CAT)
        if not is_cat and neighbor_is_cat:
          continue
        dest_r, dest_c = nr + dr, nc + dc
        owner = 0 if neighbor in (_P0_KITTEN, _P0_CAT) else 1
        n_type = 1 if neighbor_is_cat else 0
        if not (0 <= dest_r < _ROWS and 0 <= dest_c < _COLS):
          self.board[nr, nc] = _EMPTY
          self._hand[owner][n_type] += 1
        elif self.board[dest_r, dest_c] == _EMPTY:
          self.board[dest_r, dest_c] = neighbor
          self.board[nr, nc] = _EMPTY

  def _promote_kittens(self, player):
    kitten_val = _KITTEN_VAL[player]
    cat_val = _CAT_VAL[player]
    to_promote = set()
    for r in range(_ROWS):
      for c in range(_COLS):
        for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
          cells = []
          for k in range(3):
            nr, nc = r + dr * k, c + dc * k
            if (0 <= nr < _ROWS and 0 <= nc < _COLS
                and self.board[nr, nc] == kitten_val):
              cells.append((nr, nc))
            else:
              break
          if len(cells) == 3:
            to_promote.update(cells)
    if not to_promote:
      return
    n = len(to_promote)
    cats_on_board = int(np.sum(self.board == cat_val))
    for pr, pc in to_promote:
      self.board[pr, pc] = _EMPTY
      self._hand[player][0] += 1
    cats_to_add = min(n, max(0, _MAX_CATS - cats_on_board - self._hand[player][1]))
    self._hand[player][1] += cats_to_add

  def _check_win(self, player):
    cat_val = _CAT_VAL[player]
    for r in range(_ROWS):
      for c in range(_COLS):
        for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
          if all(
              0 <= r + dr * k < _ROWS
              and 0 <= c + dc * k < _COLS
              and self.board[r + dr * k, c + dc * k] == cat_val
              for k in range(3)):
            return True
    return False


class BoopObserver:
  def __init__(self, params):
    if params:
      raise ValueError(f'Observation parameters not supported; passed {params}')
    board_size = 5 * _ROWS * _COLS
    self.tensor = np.zeros(board_size + 4, np.float32)
    self.dict = {
        'observation': np.reshape(self.tensor[:board_size], (5, _ROWS, _COLS)),
        'hand': self.tensor[board_size:],
    }

  def set_from(self, state, player):
    self.tensor.fill(0)
    obs = self.dict['observation']
    hand = self.dict['hand']
    opp = 1 - player
    mk, mc = _KITTEN_VAL[player], _CAT_VAL[player]
    ok, oc = _KITTEN_VAL[opp], _CAT_VAL[opp]
    for r in range(_ROWS):
      for c in range(_COLS):
        v = state.board[r, c]
        if v == _EMPTY:   obs[0, r, c] = 1.0
        elif v == mk:     obs[1, r, c] = 1.0
        elif v == mc:     obs[2, r, c] = 1.0
        elif v == ok:     obs[3, r, c] = 1.0
        elif v == oc:     obs[4, r, c] = 1.0
    hand[0] = state._hand[player][0] / _MAX_KITTENS
    hand[1] = state._hand[player][1] / _MAX_CATS
    hand[2] = state._hand[opp][0] / _MAX_KITTENS
    hand[3] = state._hand[opp][1] / _MAX_CATS

  def string_from(self, state, player):
    del player
    return str(state)


pyspiel.register_game(_GAME_TYPE, BoopGame)

In [ ]:
game = pyspiel.load_game('python_boop')
state = game.new_initial_state()
print('Game:', game.get_type().long_name)
print('Num distinct actions:', game.num_distinct_actions())
print('Observation tensor size:', game.observation_tensor_size())
print()
print('Initial state:')
print(state)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import random

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


class PolicyValueNet(nn.Module):
    '''Shared-trunk network with separate policy and value heads.

    Policy head: logits over all actions (cross-entropy vs MCTS visit counts)
    Value head: scalar in [-1, 1] (MSE vs game outcome from current player POV)
    '''
    def __init__(self, obs_size, num_actions, hidden=(256, 256, 256)):
        super().__init__()
        layers = []
        in_size = obs_size
        for h in hidden:
            layers += [nn.Linear(in_size, h), nn.LayerNorm(h), nn.ReLU()]
            in_size = h
        self.trunk = nn.Sequential(*layers)
        self.policy_head = nn.Linear(in_size, num_actions)
        self.value_head = nn.Sequential(
            nn.Linear(in_size, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Tanh())

    def forward(self, x):
        h = self.trunk(x)
        return self.policy_head(h), self.value_head(h)

print(f'Using device: {DEVICE}')

In [ ]:
from open_spiel.python.algorithms import mcts as mcts_lib


class NNEvaluator(mcts_lib.Evaluator):
    '''Wraps PolicyValueNet as an OpenSpiel MCTS evaluator.

    prior()    -> (action, prob) pairs from policy head  [called at node expansion]
    evaluate() -> list of returns per player             [called at leaf nodes]
    '''
    def __init__(self, network, device=DEVICE):
        self.network = network
        self.device = device

    def _obs_tensor(self, state):
        obs = state.observation_tensor(state.current_player())
        return torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(self.device)

    def evaluate(self, state):
        with torch.no_grad():
            _, value = self.network(self._obs_tensor(state))
        v = value.item()
        # MCTS indexes returns by player; for 2-player zero-sum,
        # current player gets v, opponent gets -v
        current_player = state.current_player()
        returns = [-v, -v]
        returns[current_player] = v
        return returns

    def prior(self, state):
        with torch.no_grad():
            logits, _ = self.network(self._obs_tensor(state))
        logits = logits.squeeze().cpu().numpy()
        legal = state.legal_actions()
        if not legal:
            return []
        lg = logits[legal] - logits[legal].max()
        probs = np.exp(lg)
        probs /= probs.sum()
        return list(zip(legal, probs.tolist()))


def make_bot(game, network, num_simulations, device=DEVICE):
    '''Build an MCTSBot using the given network as the AlphaZero evaluator.'''
    return mcts_lib.MCTSBot(
        game,
        uct_c=1.4,
        max_simulations=num_simulations,
        evaluator=NNEvaluator(network, device),
        dirichlet_noise=(0.25, 0.3),  # epsilon=0.25, alpha=0.3 (as in the paper)
    )


def self_play_game(game, bot, temperature=1.0):
    '''Play one game via MCTS self-play.

    Returns list of (observation, mcts_policy_vector, outcome) per move,
    where outcome is the game return for the player who made that move.
    '''
    state = game.new_initial_state()
    history = []
    while not state.is_terminal():
        player = state.current_player()
        obs = list(state.observation_tensor(player))

        root = bot.mcts_search(state)
        legal = state.legal_actions()
        visit_map = {ch.action: ch.explore_count for ch in root.children}
        counts = np.array([visit_map.get(a, 0) for a in legal], dtype=float)

        if temperature < 1e-3 or counts.sum() == 0:
            probs = np.zeros(len(legal))
            probs[counts.argmax()] = 1.0
        else:
            ct = counts ** (1.0 / temperature)
            probs = ct / ct.sum()

        policy_vec = np.zeros(game.num_distinct_actions())
        for a, p in zip(legal, probs):
            policy_vec[a] = p

        action = np.random.choice(legal, p=probs)
        history.append((player, obs, policy_vec))
        state.apply_action(action)

    returns = state.returns()
    return [(obs, pol, returns[pl]) for pl, obs, pol in history]


def greedy_action(network, state, device=DEVICE):
    '''Pick the legal action with the highest policy logit (no MCTS).'''
    obs = torch.tensor(
        state.observation_tensor(state.current_player()),
        dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, _ = network(obs)
    legal = state.legal_actions()
    return legal[int(logits.squeeze().cpu().numpy()[legal].argmax())]


def eval_vs_random(game, network, num_games=100, device=DEVICE):
    '''Win rate of the greedy policy (P0) vs a random opponent (P1).'''
    wins = 0
    for _ in range(num_games):
        state = game.new_initial_state()
        while not state.is_terminal():
            if state.current_player() == 0:
                action = greedy_action(network, state, device)
            else:
                action = random.choice(state.legal_actions())
            state.apply_action(action)
        if state.returns()[0] > 0:
            wins += 1
    return wins / num_games


def eval_vs_snapshot(game, network, snapshot, num_games=100, device=DEVICE):
    '''Win rate of the current network (P0) vs an older snapshot (P1).'''
    wins = 0
    for _ in range(num_games):
        state = game.new_initial_state()
        while not state.is_terminal():
            net = network if state.current_player() == 0 else snapshot
            action = greedy_action(net, state, device)
            state.apply_action(action)
        if state.returns()[0] > 0:
            wins += 1
    return wins / num_games


def update_elo(elo, opp_elo, win_rate, k=32):
    expected = 1.0 / (1.0 + 10 ** ((opp_elo - elo) / 400.0))
    return elo + k * (win_rate - expected)

In [ ]:
# ---- Hyperparameters ----
# Runtime estimate: ~30-90 min on Colab GPU depending on MCTS_SIMS and game length.
# Raise MCTS_SIMS for stronger play (at the cost of speed).
NUM_EPISODES = 1_000      # self-play games (quality >> quantity vs DQN)
MCTS_SIMS = 50            # MCTS simulations per move
EVAL_EVERY = 100          # evaluation frequency (episodes)
EVAL_GAMES = 100          # games per evaluation
BATCH_SIZE = 256
TRAIN_STEPS_PER_EP = 4   # gradient updates per episode
MAX_BUFFER = 100_000      # replay buffer capacity
HIDDEN = (256, 256, 256)
LR = 1e-3
RANDOM_ELO = 600.0

# ---- Setup ----
game = pyspiel.load_game('python_boop')
info_state_size = game.observation_tensor_size()  # 184
num_actions = game.num_distinct_actions()         # 72

network = PolicyValueNet(info_state_size, num_actions, HIDDEN).to(DEVICE)
optimizer = torch.optim.Adam(network.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPISODES)

# Single bot reuses the network by reference — weights update automatically
bot = make_bot(game, network, MCTS_SIMS, DEVICE)

replay_buffer = []
elo = 1000.0
snapshot = copy.deepcopy(network).to(DEVICE)
history = {
    'ep': [], 'policy_loss': [], 'value_loss': [],
    'vs_random': [], 'vs_prev': [], 'elo': []
}

print(f'Device: {DEVICE}  |  obs_size={info_state_size}  |  actions={num_actions}')
print(f'{NUM_EPISODES} episodes, {MCTS_SIMS} MCTS sims/move, eval every {EVAL_EVERY}')
print('-' * 72)

for ep in range(1, NUM_EPISODES + 1):

    # 1. Self-play: generate training data with MCTS
    network.eval()
    game_data = self_play_game(game, bot, temperature=1.0)
    replay_buffer.extend(game_data)
    if len(replay_buffer) > MAX_BUFFER:
        del replay_buffer[:-MAX_BUFFER]

    # 2. Train on random minibatches from the replay buffer
    network.train()
    p_losses, v_losses = [], []
    if len(replay_buffer) >= BATCH_SIZE:
        for _ in range(TRAIN_STEPS_PER_EP):
            batch = random.sample(replay_buffer, BATCH_SIZE)
            obs_b = torch.from_numpy(np.array([x[0] for x in batch], dtype=np.float32)).to(DEVICE)
            pol_b = torch.from_numpy(np.array([x[1] for x in batch], dtype=np.float32)).to(DEVICE)
            val_b = torch.from_numpy(np.array([[x[2]] for x in batch], dtype=np.float32)).to(DEVICE)

            logits, value = network(obs_b)
            # Cross-entropy with soft MCTS policy targets
            p_loss = -(pol_b * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            # MSE value loss; tanh output and outcomes are both in [-1, 1]
            v_loss = F.mse_loss(value, val_b)
            loss = p_loss + v_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            p_losses.append(p_loss.item())
            v_losses.append(v_loss.item())
    scheduler.step()

    # 3. Periodic evaluation
    if ep % EVAL_EVERY == 0:
        network.eval()
        wr_rand = eval_vs_random(game, network, EVAL_GAMES, DEVICE)
        wr_prev = eval_vs_snapshot(game, network, snapshot, EVAL_GAMES, DEVICE)
        elo = update_elo(elo, RANDOM_ELO, wr_rand)
        snapshot = copy.deepcopy(network).to(DEVICE)

        mean_p = sum(p_losses) / len(p_losses) if p_losses else float('nan')
        mean_v = sum(v_losses) / len(v_losses) if v_losses else float('nan')
        history['ep'].append(ep)
        history['policy_loss'].append(mean_p)
        history['value_loss'].append(mean_v)
        history['vs_random'].append(wr_rand)
        history['vs_prev'].append(wr_prev)
        history['elo'].append(elo)

        print(f'Ep {ep:5d} | p_loss={mean_p:.3f} v_loss={mean_v:.3f} | '
              f'vs_random={wr_rand:.2f} | vs_prev={wr_prev:.2f} | ELO={elo:.0f}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Boop AlphaZero Training', fontsize=14)

axes[0, 0].plot(history['ep'], history['policy_loss'])
axes[0, 0].set_title('Policy Loss (cross-entropy)')
axes[0, 0].set_xlabel('Episode')

axes[0, 1].plot(history['ep'], history['value_loss'], color='tab:orange')
axes[0, 1].set_title('Value Loss (MSE)')
axes[0, 1].set_xlabel('Episode')

axes[0, 2].plot(history['ep'], history['vs_random'], color='tab:green')
axes[0, 2].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[0, 2].set_title('Win Rate vs Random (greedy)')
axes[0, 2].set_xlabel('Episode')
axes[0, 2].set_ylim(0, 1)

axes[1, 0].plot(history['ep'], history['vs_prev'], color='tab:purple')
axes[1, 0].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[1, 0].set_title('Win Rate vs Previous Checkpoint')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylim(0, 1)

axes[1, 1].plot(history['ep'], history['elo'], color='tab:red')
axes[1, 1].axhline(RANDOM_ELO, color='gray', linestyle='--', linewidth=0.8,
                    label=f'Random baseline ({RANDOM_ELO:.0f})')
axes[1, 1].set_title('ELO Rating')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].legend()

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()